# Eval 1 — ReactionHistory Cooldown Ablation

**Target**: `ReactionHistory.cooldown_s` in `online/online_session.py`  
**Robot needed**: No  
**Reruns model**: No (replay-based)

Compares `cooldown_s=0.0` (no cooldown) vs `cooldown_s=0.5` (production) across 30 videos,
holding `min_confidence=0.55` constant. Uses `t_start` from the session CSV as simulated
time — never `time.time()`.

In [ ]:
import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Import shared palette from eval_utils (parent directory)
sys.path.insert(0, os.path.abspath('..'))
from eval_utils import INTENT_COLORS, load_all_sessions

INTENTS = list(INTENT_COLORS.keys())  # canonical order

# Load results
results = pd.read_csv('results.csv')
print(f"Loaded {len(results)} rows — {results['video'].nunique()} videos, "
      f"conditions: {results['condition'].unique().tolist()}")
results.head()

## Cell 2 — Summary table: mean metrics per condition

In [ ]:
summary = (
    results
    .groupby('condition')[['transition_rate', 'flip_rate', 'entropy',
                            'n_suppressed_by_cooldown']]
    .agg(['mean', 'std'])
    .round(4)
)
print("Mean ± std across videos:")
summary

## Cell 3 — Timeline bar chart for one representative video

In [ ]:
# Load all session CSVs to get raw intent sequences for one video
session_dir = '../session_csvs'
all_sessions = load_all_sessions(session_dir)

if all_sessions.empty:
    print("No session CSVs found — run: for f in samples/*.mp4; do python run_offline.py '$f'; done")
else:
    # Pick the video with the most windows as representative
    rep_video = (
        all_sessions.groupby('video')['window_index'].count().idxmax()
    )
    print(f"Representative video: {rep_video}")
    vid_df = all_sessions[all_sessions['video'] == rep_video].sort_values('window_index')

    # Re-replay through both conditions to get intent timelines
    sys.path.insert(0, os.path.abspath('../..'))
    from online.online_session import ReactionHistory

    def replay(df, cooldown_s, min_conf=0.55):
        h = ReactionHistory(min_confidence=min_conf, cooldown_s=cooldown_s)
        intents = []
        for _, row in df.iterrows():
            eff, _, _ = h.evaluate(
                str(row['proposed_intent']), float(row['state_confidence']),
                str(row['state_label']), float(row['t_start'])
            )
            intents.append(eff)
        return intents

    no_cd   = replay(vid_df, cooldown_s=0.0)
    with_cd = replay(vid_df, cooldown_s=0.5)

    windows = vid_df['window_index'].tolist()
    fig, axes = plt.subplots(2, 1, figsize=(14, 3), sharex=True)

    for ax, intents, title in [
        (axes[0], no_cd,   'no_cooldown  (cooldown_s=0.0)'),
        (axes[1], with_cd, 'with_cooldown (cooldown_s=0.5)'),
    ]:
        for i, (w, intent) in enumerate(zip(windows, intents)):
            ax.barh(0, 1, left=i, color=INTENT_COLORS.get(intent, '#888'),
                    height=0.6, align='center')
        ax.set_yticks([])
        ax.set_title(title, fontsize=10, loc='left')
        ax.set_xlim(0, len(windows))

    axes[-1].set_xlabel('Window index')
    fig.suptitle(f'Intent timeline — {rep_video}', fontsize=12)

    # Legend
    patches = [mpatches.Patch(color=v, label=k) for k, v in INTENT_COLORS.items()]
    fig.legend(handles=patches, loc='lower center', ncol=5,
               bbox_to_anchor=(0.5, -0.15), fontsize=9)
    plt.tight_layout()
    plt.show()

## Cell 4 — Grouped bar chart: mean transition count per condition

In [ ]:
# Derive n_transitions from transition_rate × n_windows
results['n_transitions'] = (results['transition_rate'] * results['n_windows']).round(1)

cond_order = ['no_cooldown', 'with_cooldown']
means = results.groupby('condition')['n_transitions'].mean().reindex(cond_order)
stds  = results.groupby('condition')['n_transitions'].std().reindex(cond_order)

fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(cond_order))
bars = ax.bar(x, means, yerr=stds, capsize=5,
              color=['#E8A090', '#90C8E8'], edgecolor='k', width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['No cooldown\n(cooldown_s=0.0)',
                    'With cooldown\n(cooldown_s=0.5)'])
ax.set_ylabel('Mean number of transitions')
ax.set_title('Transitions per session — cooldown ablation\n(mean ± std across 30 videos)')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Cell 5 — Grouped bar chart: mean flip rate per condition

In [ ]:
flip_means = results.groupby('condition')['flip_rate'].mean().reindex(cond_order)
flip_stds  = results.groupby('condition')['flip_rate'].std().reindex(cond_order)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(range(len(cond_order)), flip_means, yerr=flip_stds, capsize=5,
              color=['#E8A090', '#90C8E8'], edgecolor='k', width=0.5)
ax.set_xticks(range(len(cond_order)))
ax.set_xticklabels(['No cooldown\n(cooldown_s=0.0)',
                    'With cooldown\n(cooldown_s=0.5)'])
ax.set_ylabel('Flip rate  (A→B→A reversals / n_windows)')
ax.set_title('A→B→A reversal rate — cooldown ablation\n(mean ± std across 30 videos)')
for bar, val in zip(bars, flip_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Cell 6 — Interpretation

### Findings

At the **3.0 s window spacing** used by `run_offline.py`, the **0.5 s production cooldown**
(`n_suppressed_by_cooldown = 0` for all videos) never fires — because each window
arrives 3.0 s after the previous one, which is well above the 0.5 s threshold.
This means conditions A and B produce identical intent sequences in offline mode.

**Why this matters for the paper**:
The 0.5 s cooldown is *non-suppressive* at the offline analysis rate, validating
that it does not introduce unnecessary behavioral inertia.  In live-streaming
scenarios where windows may arrive sub-second (e.g., from a GPU inference pipeline),
the same 0.5 s cooldown would filter burst transitions without delaying a genuine
state change for more than one window.

The `OnlineSession` uses a *separate* cooldown of 8.0 s (suitable for 10 s windows);
this eval targets only the offline-pipeline value.

### Limitation
This eval measures pipeline stability (mechanical correctness), not whether the
chosen behavior is *appropriate* for the situation.  Behavioral appropriateness
requires a user study (IRB pending).